<a href="https://colab.research.google.com/github/jyryu3161/chembigdata/blob/main/lec10_BBBP_KNN_%E1%84%89%E1%85%B5%E1%86%AF%E1%84%89%E1%85%B3%E1%86%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## BBBP 혈액뇌장벽 투과 예측 — KNN 실습

**과제**: SMILES로 주어진 소분자가 혈액뇌장벽(BBB)을 투과하는지(`target=1`) 예측하고, 양성 클래스 확률(0~1)을 제출합니다.

| 항목 | 내용 |
|---|---|
| 데이터 | TDC ADMET Benchmark · BBB_Martins |
| 분할 | scaffold split · train 1,421 / val 203 / test 406 |
| 순위 지표 | **ROC-AUC** (val, test 각각) |
| 참고 지표 | AUPRC, Log Loss, Accuracy, Balanced Accuracy, F1, MCC, Precision, Recall, Specificity |
| 제출 | `sample_id, prediction` (prediction = 투과 확률) |

### 이 노트북에서 배우는 것

1. SMILES → **분자 지문(fingerprint)** / **descriptor** 로 바꾸기
2. **KNN(k-최근접 이웃)** 분류기의 원리와 하이퍼파라미터 (`k`, 거리, 가중치)
3. 정답이 없는 val/test 대신 **scaffold 기반 교차검증**으로 성능을 추정하는 법
4. 불균형 데이터에서 여러 지표를 함께 읽는 법
5. 제출 파일 만들기

> ⚠️ **규칙**: 공개된 BBBP 정답(MoleculeNet, TDC 원본 등)을 조회해 제출하는 것은 부정행위입니다. 이 노트북은 주어진 `train.csv`의 라벨만 사용합니다.

셀은 위에서부터 순서대로 실행하세요.

## 0. 환경 준비

Colab에는 RDKit이 기본 설치되어 있지 않으므로 한 번 설치합니다. (scikit-learn, pandas, matplotlib은 기본 포함)

In [ ]:
!pip install -q rdkit

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, Draw, rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, log_loss,
                             accuracy_score, balanced_accuracy_score, f1_score,
                             matthews_corrcoef, precision_score, recall_score,
                             confusion_matrix, roc_curve)

RDLogger.DisableLog("rdApp.*")
SEED = 42

## 1. 데이터 불러오기

대회 파일 4개(`train.csv`, `val.csv`, `test.csv`, `sample_submission.csv`)가 필요합니다.

- 아래 셀을 실행하면 파일 선택 창이 뜹니다. 4개 파일을 한꺼번에 선택해 업로드하세요.

In [ ]:
import os

DATA_DIR = "."
FILES = ["train.csv", "val.csv", "test.csv", "sample_submission.csv"]

missing = [f for f in FILES if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    from google.colab import files
    print("업로드할 파일:", missing)
    files.upload()

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print("train:", train.shape, "| val:", val.shape, "| test:", test.shape, "| sample:", sample_sub.shape)
train.head(10)

**확인할 점**: `train`에만 `target` 열이 있고, `val`/`test`에는 SMILES만 있습니다.
즉, val/test의 ROC-AUC는 제출 후에야 알 수 있으므로 **모델 선택은 train 안에서 검증**해야 합니다 (4장).

In [ ]:
ratio = train["target"].value_counts().sort_index()
print(ratio)
print(f"양성(BBB+) 비율: {train['target'].mean():.1%}")

ratio.plot.bar(color=["tab:red", "tab:blue"])
plt.xticks([0, 1], ["BBB− (0)", "BBB+ (1)"], rotation=0)
plt.ylabel("Count")
plt.title("Train class balance")
plt.show()

양성이 약 3/4입니다. **모든 분자를 1로 찍어도 Accuracy ≈ 76%** 가 나오므로, Accuracy만 보면 안 됩니다.
그래서 이 대회는 임계값(0.5)과 무관한 **ROC-AUC**로 순위를 매기고, Balanced Accuracy·MCC 등을 함께 보라고 안내합니다.

### SMILES 유효성 검사
RDKit이 읽지 못하는 SMILES가 있으면 특징을 만들 수 없습니다. 미리 확인합니다.

In [ ]:
for name, df in [("train", train), ("val", val), ("test", test)]:
    df["mol"] = df["SMILES"].apply(Chem.MolFromSmiles)
    print(f"{name}: 파싱 실패 {df['mol'].isna().sum()}개 / {len(df)}개")

In [ ]:
# 클래스별 예시 분자 보기
examples = pd.concat([train[train.target == 1].head(4), train[train.target == 0].head(4)])
Draw.MolsToGridImage(examples["mol"].tolist(), molsPerRow=4, subImgSize=(220, 180),
                     legends=["BBB+ (target=1)" if t == 1 else "BBB- (target=0)" for t in examples["target"]])

## 2. 분자를 숫자 벡터로 바꾸기 (Featurization)

KNN은 "**가까운** 분자들의 라벨을 보고 투표"하는 모델이라, **분자 사이의 거리**를 정의할 수 있어야 합니다.
두 가지 표현을 비교해 봅니다.

| 표현 | 내용 | 거리계산 지표 |
|---|---|---|
| **Morgan fingerprint (ECFP4)** | 원자 주변 반경 2의 부분구조 존재 여부를 2048비트로 표시 | **Jaccard(=1−Tanimoto)** |
| **RDKit descriptor** | 분자량, logP, TPSA, H-결합 주개/받개 수 등 물리화학적 수치 | 표준화 후 Euclidean |

> 💡 BBB 투과는 **TPSA(극성 표면적)**, **H-결합 주개 수**, **logP**, **분자량** 등과 관련이 크다고 알려져 있습니다 (CNS 약물 규칙).
> 그래서 descriptor 기반 KNN도 꽤 경쟁력이 있을 수 있습니다. 직접 비교해 봅시다.

In [ ]:
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def morgan_fp(mols):
    # Jaccard 거리를 쓰려면 bool 배열이어야 합니다
    return np.array([morgan_gen.GetFingerprintAsNumPy(m) for m in mols], dtype=bool)

X_train_fp = morgan_fp(train["mol"])
X_val_fp = morgan_fp(val["mol"])
X_test_fp = morgan_fp(test["mol"])
y_train = train["target"].values

print("fingerprint shape:", X_train_fp.shape)
print("분자당 켜진 비트 수 (평균):", X_train_fp.sum(1).mean().round(1))

In [ ]:
DESC_NAMES = ["MolWt", "MolLogP", "TPSA", "NumHDonors", "NumHAcceptors",
              "NumRotatableBonds", "RingCount", "NumAromaticRings",
              "HeavyAtomCount", "FractionCSP3", "NHOHCount", "NOCount"]
DESC_FUNCS = dict(Descriptors.descList)

def rdkit_desc(mols):
    return np.array([[DESC_FUNCS[n](m) for n in DESC_NAMES] for m in mols])

X_train_desc = rdkit_desc(train["mol"])
X_val_desc = rdkit_desc(val["mol"])
X_test_desc = rdkit_desc(test["mol"])

print("클래스별 descriptor 평균 (0 = BBB- 비투과, 1 = BBB+ 투과)")
pd.DataFrame(X_train_desc, columns=DESC_NAMES).assign(target=y_train).groupby("target").mean().round(2).T

위 표에서 BBB+ 분자가 BBB− 분자보다 **TPSA, H-결합 주개 수(NumHDonors), 분자량이 작고 logP가 큰** 경향을 확인할 수 있습니다.
분포로 보면 더 분명합니다.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, name in zip(axes, ["TPSA", "NumHDonors", "MolWt", "MolLogP"]):
    col = DESC_NAMES.index(name)
    for t, color in [(0, "tab:red"), (1, "tab:blue")]:
        ax.hist(X_train_desc[y_train == t, col], bins=30, alpha=0.5, density=True,
                color=color, label=f"BBB{'+' if t else '−'}")
    ax.set_title(name)
axes[0].legend()
plt.tight_layout()
plt.show()

## 3. KNN 분류기 이해하기

새 분자 $x$의 투과 확률을 다음처럼 계산합니다.

1. train 분자 중 $x$와 거리가 가장 가까운 **k개**를 찾는다.
2. 그 k개 이웃 중 양성(1)의 비율 → 예측 확률
   - `weights="uniform"`: 단순 비율
   - `weights="distance"`: 가까운 이웃일수록 더 큰 가중치 ($1/d$)

**특징**
- 하이퍼파라미터: `k`, 거리(`metric`), `weights`
- 거리 스케일에 민감 → descriptor는 반드시 **표준화(StandardScaler)**
- `k`가 작으면 과적합(분산↑), 크면 과소적합(편향↑)

> 📝 `weights="uniform"`이면 확률이 0, 1/k, 2/k, … 처럼 **k+1개 값**만 나옵니다. 동점이 많아져 ROC-AUC가 조금 손해를 볼 수 있으니 `distance` 가중치와 비교해 보세요.

간단한 예로, train의 한 분자에 대한 이웃을 직접 보겠습니다.

In [ ]:
query_idx = 0
knn_demo = KNeighborsClassifier(n_neighbors=5, metric="jaccard", algorithm="brute")
knn_demo.fit(X_train_fp, y_train)

# 자기 자신(거리 0)이 결과에 섞여 나오므로, 6개를 찾은 뒤 자기 인덱스를 빼고 5개를 씁니다
dist, idx = knn_demo.kneighbors(X_train_fp[[query_idx]], n_neighbors=6)
dist, idx = dist[0][1:], idx[0][1:]

mols = [train["mol"][query_idx]] + [train["mol"][i] for i in idx]
legends = [f"QUERY (true={y_train[query_idx]})"] + \
          [f"Tanimoto={1-d:.2f}, y={y_train[i]}" for d, i in zip(dist, idx)]
print(f"이웃 5개 중 양성 비율 = {y_train[idx].mean():.2f}  →  uniform KNN의 예측 확률")
Draw.MolsToGridImage(mols, molsPerRow=6, subImgSize=(200, 180), legends=legends)

## 4. 검증 전략: Scaffold 기반 교차검증

val/test 정답이 없으므로 train을 나눠서 성능을 추정합니다. 그런데 **어떻게 나누느냐**가 중요합니다.

- 이 대회의 val/test는 **scaffold split**: 분자의 핵심 골격(Bemis–Murcko scaffold)이 train과 겹치지 않도록 나뉘어 있습니다.
- 무작위로 나누면 같은 골격의 분자가 train/검증 양쪽에 들어가서, 특히 **KNN처럼 이웃을 찾는 모델은 성능이 과대평가**됩니다.

그래서 scaffold를 그룹으로 두고 `StratifiedGroupKFold`(같은 scaffold는 같은 fold, 클래스 비율은 유지)로 5-fold CV를 합니다.

In [ ]:
def murcko_scaffold(mol):
    return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)

train["scaffold"] = train["mol"].apply(murcko_scaffold)
print("고유 scaffold 수:", train["scaffold"].nunique(), "/ train 분자 수:", len(train))
train["scaffold"].value_counts().head(10)

### 평가 함수
대회의 순위 지표와 참고 지표를 모두 계산하는 함수를 만들어 둡니다. 라벨 지표는 대회와 같게 **임계값 0.5**로 계산합니다.

In [ ]:
def evaluate(y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "ROC-AUC": roc_auc_score(y_true, prob),
        "AUPRC": average_precision_score(y_true, prob),
        "LogLoss": log_loss(y_true, np.clip(prob, 1e-6, 1 - 1e-6)),
        "Accuracy": accuracy_score(y_true, pred),
        "BalancedAcc": balanced_accuracy_score(y_true, pred),
        "F1": f1_score(y_true, pred, zero_division=0),
        "MCC": matthews_corrcoef(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "Specificity": tn / (tn + fp),
    }

In [ ]:
# 5-fold cross validation
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(cv.split(X_train_fp, y_train, groups=train["scaffold"]))

def cv_predict(make_model, X, y=y_train):
    # 4개 fold로 학습 → 남은 1개 fold를 예측, 이를 5번 반복하면 모든 분자가 한 번씩 예측됨
    # 각 예측은 그 분자를 학습에 안 쓴 모델이 만든 것 = out-of-fold(OOF) 예측
    oof = np.zeros(len(y))
    for tr_idx, va_idx in folds:
        model = make_model().fit(X[tr_idx], y[tr_idx])
        oof[va_idx] = model.predict_proba(X[va_idx])[:, 1]
    return oof

for i, (tr_idx, va_idx) in enumerate(folds):
    print(f"fold {i+1}: 학습용 {len(tr_idx)}개, 검증용 {len(va_idx)}개, 검증용 투과(1)분자 비율 {y_train[va_idx].mean():.2f}")

검증용 투과분자 비율이 train 전체 비율(76%)와 유사합니다. Stratified가 fold마다 클래스 비율을 맞춰 준 결과입니다.

### Baseline: KNN (k=5)
먼저 기본 설정으로 두 표현을 비교합니다.

In [ ]:
def knn_fp(k=5, weights="uniform"):
    return KNeighborsClassifier(n_neighbors=k, weights=weights, metric="jaccard", algorithm="brute")

def knn_desc(k=5, weights="uniform"):
    return make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k, weights=weights))

baseline = pd.DataFrame({
    "Morgan + Jaccard (k=5)": evaluate(y_train, cv_predict(knn_fp, X_train_fp)),
    "Descriptor + Euclidean (k=5)": evaluate(y_train, cv_predict(knn_desc, X_train_desc)),
}).T.round(3)
baseline

**지표 정리** (양성 = 투과 분자, 음성 = 비투과 분자)

| 지표 | 한 줄 의미 |
|---|---|
| ROC-AUC | 투과 분자에 비투과 분자보다 **높은 점수**를 주는가? (0.5 = 찍기, 1 = 완벽) |
| AUPRC | "투과"라고 높게 찍은 분자들이 정말 투과인지? (찍기 = 0.76) |
| LogLoss | 확률이 정답과 얼마나 먼지? **낮을수록 좋음**, 확신하고 틀리면 크게 깎임 |
| Accuracy | 전체 중 몇 개 맞혔는지? (전부 "투과"라고만 해도 0.76) |
| BalancedAcc | 투과 맞힌 비율과 비투과 맞힌 비율의 평균 |
| F1 | Precision과 Recall을 함께 본 점수 |
| MCC | 양쪽 클래스를 모두 고려한 종합 점수 (0 = 찍기, 1 = 완벽) |
| Precision | "투과"라고 한 것 중 진짜 투과는 몇 %인지? |
| Recall | 진짜 투과 분자 중 몇 %를 찾아냈는지? |
| Specificity | 진짜 비투과 분자 중 몇 %를 찾아냈는지? |

> Accuracy 아래 지표들은 확률 0.5 이상을 "투과"로 판정한 뒤 계산합니다.

## 5. 하이퍼파라미터 탐색: k와 가중치

`k`를 1~50 사이에서 바꿔 가며 scaffold CV ROC-AUC를 그려 봅니다.

In [ ]:
K_LIST = [1, 3, 5, 7, 9, 11, 15, 21, 31, 41, 51]
rows = []
for k in K_LIST:
    for w in ["uniform", "distance"]:
        rows.append({"feature": "Morgan", "k": k, "weights": w,
                     "ROC-AUC": roc_auc_score(y_train, cv_predict(lambda: knn_fp(k, w), X_train_fp))})
        rows.append({"feature": "Descriptor", "k": k, "weights": w,
                     "ROC-AUC": roc_auc_score(y_train, cv_predict(lambda: knn_desc(k, w), X_train_desc))})
grid = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 4.5))
for (feat, w), g in grid.groupby(["feature", "weights"]):
    ax.plot(g["k"], g["ROC-AUC"], marker="o", ls="-" if w == "distance" else "--", label=f"{feat} / {w}")
ax.set_xlabel("k (number of neighbors)")
ax.set_ylabel("scaffold-CV ROC-AUC")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

grid.sort_values("ROC-AUC", ascending=False).head(8).round(4)

**그래프 읽는 법**
- k=1은 가장 가까운 한 분자만 따라 하므로 확률이 0 또는 1뿐 → AUC가 낮습니다.
- k를 키우면 예측이 매끄러워지며 AUC가 오르다가, 너무 크면 먼 분자까지 섞여 다시 떨어지거나 평평해집니다.
- 실선(`distance`)과 점선(`uniform`)의 차이도 확인해 보세요.

## 6. 최종 모델 선택

scaffold CV ROC-AUC가 가장 높았던 설정을 최종 모델로 씁니다.
baseline(k=5, uniform)과 나란히 놓고 무엇이 달라졌는지 확인합니다.

In [ ]:
best = grid[grid.feature == "Morgan"].sort_values("ROC-AUC", ascending=False).iloc[0]
BEST_K, BEST_W = int(best.k), best.weights
print(f"선택한 설정: Morgan / k={BEST_K} / weights={BEST_W}  (CV ROC-AUC {best['ROC-AUC']:.3f})")

make_final = lambda: knn_fp(BEST_K, BEST_W)
oof_base = cv_predict(knn_fp, X_train_fp)
oof_final = cv_predict(make_final, X_train_fp)

print("scaffold CV 성능 비교: baseline vs 최종 모델")
pd.DataFrame({
    "Baseline (k=5, uniform)": evaluate(y_train, oof_base),
    f"Final (k={BEST_K}, {BEST_W})": evaluate(y_train, oof_final),
}).T.round(3)

**결과**
- **ROC-AUC**는 오르고, **LogLoss**는 줄었습니다. 이웃을 더 많이 보고(k↑) 가까운 이웃에 가중치를 주니 확률이 0과 1에 덜 몰립니다.
- 반면 **Specificity**는 떨어졌습니다. 이웃이 많아지면 다수 클래스(투과)가 더 많이 섞여 확률이 1 쪽으로 밀리기 때문입니다. 순위(ROC-AUC)는 좋아져도 0.5 기준 판정은 더 "투과" 쪽으로 치우칩니다.
- ⚠️ k와 weights를 이 CV 점수로 고르고 같은 점수로 보고하므로 약간 낙관적인 값입니다. 실제 val/test 점수는 이보다 낮을 수 있습니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, p in [("Baseline (k=5, uniform)", oof_base), (f"Final (k={BEST_K}, {BEST_W})", oof_final)]:
    fpr, tpr, _ = roc_curve(y_train, p)
    axes[0].plot(fpr, tpr, label=f"{name}  AUC={roc_auc_score(y_train, p):.3f}")
axes[0].plot([0, 1], [0, 1], "k:", lw=1)
axes[0].set_xlabel("False Positive Rate (1 - Specificity)")
axes[0].set_ylabel("True Positive Rate (Recall)")
axes[0].set_title("ROC curve (scaffold-CV, OOF)")
axes[0].legend()

for t, color in [(0, "tab:red"), (1, "tab:blue")]:
    axes[1].hist(oof_final[y_train == t], bins=20, range=(0, 1), alpha=0.6, color=color,
                 label=f"true BBB{'+' if t else '-'} (target={t})")
axes[1].axvline(0.5, color="k", ls="--", lw=1)
axes[1].set_xlabel("predicted probability (Final model)")
axes[1].set_title("OOF probability distribution")
axes[1].legend()
plt.tight_layout()
plt.show()

**그래프 읽기**
- 왼쪽 ROC 곡선: 곡선이 왼쪽 위로 붙을수록 좋은 모델이고, 곡선 아래 면적이 ROC-AUC입니다.
- 오른쪽 분포: 빨강(실제 비투과)은 0 쪽, 파랑(실제 투과)은 1 쪽에 몰릴수록 잘 구분한 것입니다. 점선(0.5) **오른쪽의 빨간 막대**가 "투과"로 잘못 판정된 비투과 분자이고, 이것이 많을수록 Specificity가 낮아집니다.

## 7. 최종 모델 학습 & 제출 파일 만들기

CV로 고른 설정으로 **train 전체**를 다시 학습하고, val과 test를 예측합니다.
제출 파일의 행 순서는 `sample_submission.csv`와 같게 맞춥니다.

In [ ]:
final_model = make_final().fit(X_train_fp, y_train)

pred = pd.concat([
    pd.DataFrame({"sample_id": val["sample_id"], "prediction": final_model.predict_proba(X_val_fp)[:, 1]}),
    pd.DataFrame({"sample_id": test["sample_id"], "prediction": final_model.predict_proba(X_test_fp)[:, 1]}),
])

submission = sample_sub[["sample_id"]].merge(pred, on="sample_id", how="left")

# 제출 전 점검
assert len(submission) == len(sample_sub)
assert submission["prediction"].notna().all()
assert submission["prediction"].between(0, 1).all()

submission.to_csv("submission_knn.csv", index=False)
print(submission.shape)
submission.head()

In [ ]:
# val / test 예측 확률 분포 확인 (양성 비율: val 약 70%, test 약 81%로 안내됨)
for prefix in ["val", "test"]:
    p = submission.loc[submission.sample_id.str.startswith(prefix), "prediction"]
    print(f"{prefix}: n={len(p)}, 평균 확률={p.mean():.3f}, 0.5 이상 비율={(p >= 0.5).mean():.1%}")

**결과 읽기**
- 0.5 이상(=BBB+로 판정)인 비율이 대회에 안내된 실제 양성 비율(val 약 70%, test 약 81%)보다 높습니다. CV에서 본 양성 쪽 치우침(낮은 Specificity)이 val/test에서도 나타날 것으로 예상할 수 있습니다.
- 평균 확률은 test가 val보다 높습니다. test에 양성이 더 많다는 안내와 방향이 같습니다.
- 순위는 ROC-AUC로 매기므로 이 치우침이 순위 점수를 직접 깎지는 않습니다. 다만 Accuracy, Specificity 같은 참고 지표에는 영향을 줍니다.

In [ ]:
# Colab이면 파일을 내 컴퓨터로 내려받기
try:
    from google.colab import files
    files.download("submission_knn.csv")
except ImportError:
    print("로컬 환경: submission_knn.csv 가 현재 폴더에 저장되었습니다.")

## 8. 정리

| 단계 | 핵심 |
|---|---|
| 표현 | Morgan FP(구조) + Jaccard 거리 / Descriptor(물성) + 표준화 Euclidean |
| 모델 | KNN: k, weights, metric |
| 검증 | val/test가 scaffold split이므로 **scaffold group CV**로 추정 (random CV는 과대평가) |
| 지표 | 순위는 ROC-AUC, 불균형이므로 AUPRC·BalancedAcc·MCC·Specificity 함께 확인 |
